# V2 策略详细拆解：ML风控 + 多因子动量 + 连续仓位调节

## 目录

1. **策略总览与架构**  
2. **参数体系详解**  
3. **模块一：ML市场状态预测（风控核心）**  
   - 3.1 特征工程：反转+状态切换特征  
   - 3.2 目标变量：未来5日收益  
   - 3.3 双模型集成：HuberRegressor + HistGBR  
   - 3.4 Walk-Forward训练机制  
   - 3.5 样本加权策略  
4. **模块二：连续仓位映射**  
   - 4.1 Sigmoid映射函数  
   - 4.2 EMA平滑 + 变化率限幅  
   - 4.3 与V1三档模式的对比  
5. **模块三：多因子复合动量选股**  
   - 5.1 六因子详解  
   - 5.2 截面Z-score标准化  
   - 5.3 加权合成  
6. **模块四：危机过滤器 + ATR止损**  
   - 6.1 波动率-趋势双条件  
   - 6.2 连续衰减机制  
   - 6.3 ATR组合止损  
7. **主调仓流程：五步串联**  
8. **V1 vs V2 全面对比**  
9. **参数调优指南**

---

> **适用平台**: 聚宽(JoinQuant) 回测  
> **基准**: 沪深300 (000300.XSHG)  
> **标的池**: 20只主流ETF  
> **避险标的**: 银华日利 (511880.XSHG)

## 1. 策略总览与架构

V2策略在V1（三档风控 + 冷却期 + 趋势过滤）的基础上进行了三大升级：

| 维度 | V1 | V2 |
|:-----|:---|:---|
| **风控决策** | MA硬规则（趋势线穿越） | ML双模型集成（Walk-Forward） |
| **仓位管理** | 三档离散（100%/50%/0%） | 连续Sigmoid映射 [0,1] |
| **选股因子** | 单因子MOM | 6因子复合（MOM/ROC/RSI/MACD/趋势） |
| **仓位平滑** | 冷却期 + 确认期 | EMA平滑 + 变化率限幅 |
| **避险方式** | 三档切换到防守 | 连续调节避险标的权重 |
| **危机保护** | 趋势线穿越 | ML + 危机过滤器 + ATR止损 |

### 架构流程图

```
每个调仓日（每5个交易日）
┌──────────────────────────────────────────┐
│                                          │
│  Step 1: ML模型预测市场状态              │
│    基准指数 OHLCV → 技术特征(40+维)      │
│    → HuberRegressor + HistGBR 集成预测   │
│    → market_score ∈ ℝ                    │
│                                          │
│  Step 2: Sigmoid映射 + 危机过滤          │
│    market_score → σ(score×8) → [0,1]     │
│    × crisis_factor → raw_pct             │
│    → EMA平滑 + 限幅 → mom_pct           │
│                                          │
│  Step 3: 多因子复合动量选股              │
│    20只ETF × 6个因子 → Z-score标准化     │
│    → 加权合成 → 排名 → TopK=10          │
│                                          │
│  Step 4: 构建最终权重                    │
│    动量部分: mom_pct × 等权TopK           │
│    防守部分: (1-mom_pct) × 银华日利       │
│                                          │
│  Step 5: ATR止损检查 → 执行下单          │
└──────────────────────────────────────────┘
```

In [ ]:
# 基础依赖
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

# ML相关
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import HuberRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

# 绘图设置
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial']
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-whitegrid')

print("✅ 所有依赖加载成功")

## 2. 参数体系详解

V2策略的参数分为五个层级，下面逐一展示其含义与推荐值。

In [ ]:
# ═══════════════════════════════════════════════════════════
# 层级一：标的池与基准
# ═══════════════════════════════════════════════════════════

UNIVERSE = [
    '510300.XSHG',  # 沪深300ETF
    '510050.XSHG',  # 上证50ETF
    '510500.XSHG',  # 中证500ETF
    '159919.XSHE',  # 沪深300ETF(深)
    '159915.XSHE',  # 创业板ETF
    '588000.XSHG',  # 科创50ETF
    '588080.XSHG',  # 科创板50ETF
    '512100.XSHG',  # 中证1000ETF
    '512010.XSHG',  # 医药ETF
    '512760.XSHG',  # 半导体ETF
    '512480.XSHG',  # 半导体ETF(另)
    '512800.XSHG',  # 银行ETF
    '512660.XSHG',  # 军工ETF
    '516160.XSHG',  # 新能源ETF
    '515790.XSHG',  # 光伏ETF
    '159941.XSHE',  # 纳指ETF
    '518880.XSHG',  # 黄金ETF
    '513050.XSHG',  # 中概互联ETF
    '513100.XSHG',  # 纳斯达克ETF
    '511880.XSHG',  # 银华日利(货币ETF/避险)
]

BENCHMARK  = '000300.XSHG'   # 沪深300指数
DEFENSIVE  = '511880.XSHG'   # 避险标的

print(f"标的池: {len(UNIVERSE)} 只ETF")
print(f"基准:   {BENCHMARK}")
print(f"避险:   {DEFENSIVE}")

# ═══════════════════════════════════════════════════════════
# 层级二：选股参数
# ═══════════════════════════════════════════════════════════

TOPK             = 10     # 每次选前10名
REBALANCE_EVERY  = 5      # 每5个交易日调仓
MAX_WEIGHT       = 0.15   # 单个ETF最大权重15%

print(f"\n选股: Top-{TOPK}, 每{REBALANCE_EVERY}天调仓, 单只上限{MAX_WEIGHT:.0%}")

# ═══════════════════════════════════════════════════════════
# 层级三：多因子动量权重
# ═══════════════════════════════════════════════════════════

FACTOR_WEIGHTS = {
    'mom_20':  0.25,   # 20日收益率动量 — 中短期趋势主力
    'mom_60':  0.20,   # 60日收益率动量 — 中期趋势确认
    'roc_10':  0.15,   # 10日ROC — 短期爆发力
    'rsi_14':  0.15,   # RSI — 相对强弱（标准化后）
    'macd_h':  0.15,   # MACD柱 — 趋势加速度
    'trend':   0.10,   # EMA12-EMA26 — 趋势方向
}

print(f"\n多因子权重: {FACTOR_WEIGHTS}")
print(f"权重之和: {sum(FACTOR_WEIGHTS.values()):.2f}")

# ═══════════════════════════════════════════════════════════
# 层级四：ML风控参数
# ═══════════════════════════════════════════════════════════

ML_TRAIN_WINDOW   = 252 * 3   # 3年滚动训练窗口（756天）
ML_RETRAIN_EVERY  = 10        # 每10个交易日重新训练
ML_HORIZON        = 5         # 预测未来5日收益
ML_HALF_LIFE      = 126       # 时间衰减半衰期（约半年）
ML_TAIL_WEIGHT    = 3.0       # 极端样本加权3倍
ML_TAIL_QUANTILE  = 0.85      # 前15%极端样本

ENSEMBLE_LIN_WEIGHT  = 0.45   # 线性模型(Huber)权重
ENSEMBLE_TREE_WEIGHT = 0.55   # 树模型(HistGBR)权重

print(f"\nML风控:")
print(f"  训练窗口: {ML_TRAIN_WINDOW}天 ({ML_TRAIN_WINDOW/252:.0f}年)")
print(f"  重训频率: 每{ML_RETRAIN_EVERY}天")
print(f"  预测周期: {ML_HORIZON}天")
print(f"  集成权重: 线性{ENSEMBLE_LIN_WEIGHT:.0%} + 树{ENSEMBLE_TREE_WEIGHT:.0%}")

# ═══════════════════════════════════════════════════════════
# 层级五：仓位映射 + 危机 + 止损
# ═══════════════════════════════════════════════════════════

SIGMOID_SCALE     = 8.0    # Sigmoid缩放（越大→0/1极端化）
SIGMOID_CENTER    = 0.0    # 中心点
POS_SMOOTH_ALPHA  = 0.3    # EMA平滑系数
POS_MAX_CHANGE    = 0.25   # 单次调仓最大变化

CRISIS_VOL_RATIO_THRESH = 1.3    # 短期/长期波动率 > 1.3 = 高波动
CRISIS_TREND_THRESH     = -0.01  # EMA趋势差 < -1% = 下行
CRISIS_DAMPING          = 0.5    # 危机时仓位衰减到50%

ATR_STOP_MULTIPLIER = 3.0       # ATR倍数止损

print(f"\n仓位映射: Sigmoid(scale={SIGMOID_SCALE}), 平滑α={POS_SMOOTH_ALPHA}, 限幅={POS_MAX_CHANGE:.0%}/次")
print(f"危机过滤: vol_ratio>{CRISIS_VOL_RATIO_THRESH}, trend<{CRISIS_TREND_THRESH}, 衰减={CRISIS_DAMPING}")
print(f"ATR止损: {ATR_STOP_MULTIPLIER}×ATR")

## 3. 模块一：ML市场状态预测（风控核心）

这是V2最核心的升级。V1用简单的均线穿越判断牛熊，V2则用**机器学习双模型集成**预测基准指数未来5日收益，输出一个连续的"市场情绪得分"。

### 3.1 特征工程：反转 + 状态切换特征

ML模型的输入是基准指数（沪深300）的OHLCV数据，经过 `make_ml_features()` 提取**40+维度**的技术特征，分为6大类：

| 类别 | 特征 | 说明 |
|:-----|:-----|:-----|
| **多尺度动量** | ret_1, ret_2, ..., ret_120 | 1~120日收益率，捕捉不同周期动量 |
| **波动状态** | vol_20, vol_60, vol_ratio | 短期/长期波动率及比值，识别状态切换 |
| **反转信号** | z_price_20, rsi_14, bb_pos, bb_width | 超买超卖、均值偏离度 |
| **趋势** | trend_12_26, macd_hist, adx_14 | 趋势方向、强度、加速度 |
| **拐点检测** | d1_xxx, d3_xxx | 各指标的1阶/3阶差分，捕捉转折点 |
| **结构特征** | hl_range, co_return, price_vs_sma50/200 | 日内振幅、均线位置 |

下面我们用纯numpy/pandas实现这些特征，并可视化它们的效果。

In [ ]:
def make_ml_features(df):
    """
    构建ML特征（V2_strategy.py中的核心函数）
    
    输入: df — 含 open/high/low/close/volume 的日频DataFrame (index=日期)
    输出: df — 增加40+列技术特征
    
    特征类别:
    ├── 多尺度动量: ret_1, ret_2, ..., ret_120
    ├── 波动状态:   vol_20, vol_60, vol_ratio
    ├── 反转信号:   z_price_20, rsi_14, bb_pos, bb_width
    ├── 趋势指标:   trend_12_26, macd_hist, adx_14
    ├── 拐点检测:   d1_xxx, d3_xxx (一阶/三阶差分)
    └── 结构特征:   hl_range, co_return, price_vs_sma50/200
    """
    df = df.copy().sort_index()

    # ========== 1. 多尺度动量 ==========
    # 核心思想：不同时间尺度的收益率反映不同的市场状态
    # - 短期(1-5日): 反映短期反转/延续
    # - 中期(10-60日): 反映趋势动量
    # - 长期(120日): 反映长期趋势
    df['ret_1'] = df['close'].pct_change()
    for n in [2, 3, 5, 10, 20, 60, 120]:
        df[f'ret_{n}'] = df['close'].pct_change(n)

    # ========== 2. 波动与状态 ==========
    # vol_ratio > 1 → 短期波动升高（可能是危机前兆）
    # vol_ratio < 1 → 短期波动收敛（市场平静）
    df['vol_20'] = df['ret_1'].rolling(20).std()
    df['vol_60'] = df['ret_1'].rolling(60).std()
    df['vol_ratio'] = df['vol_20'] / (df['vol_60'] + 1e-8)

    # ========== 3. 成交量异常 ==========
    # Z-score > 2 → 异常放量（可能是趋势加速或反转）
    if 'volume' in df.columns:
        vol_mean = df['volume'].rolling(20).mean()
        vol_std = df['volume'].rolling(20).std()
        df['volu_z_20'] = (df['volume'] - vol_mean) / (vol_std + 1e-8)
    else:
        df['volu_z_20'] = 0.0

    # ========== 4. 均值偏离（Z-score）==========
    # z > 2 → 严重偏离均线（可能回归）
    # z < -2 → 严重低于均线（可能反弹）
    sma_20 = df['close'].rolling(20).mean()
    std_20 = df['close'].rolling(20).std()
    df['z_price_20'] = (df['close'] - sma_20) / (std_20 + 1e-8)

    # ========== 5. RSI ==========
    # RSI > 70 → 超买; RSI < 30 → 超卖
    delta = df['close'].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / (loss + 1e-8)
    df['rsi_14'] = 100 - 100 / (1 + rs)

    # ========== 6. 布林带位置和宽度 ==========
    # bb_pos → 价格在布林带中的相对位置 [0,1]
    # bb_width → 波动率的另一视角
    bb_upper = sma_20 + 2 * std_20
    bb_lower = sma_20 - 2 * std_20
    bb_width = bb_upper - bb_lower
    df['bb_pos'] = (df['close'] - bb_lower) / (bb_width + 1e-8)
    df['bb_width'] = bb_width / (df['close'] + 1e-8)

    # ========== 7. 趋势强度 ==========
    # EMA12 > EMA26 → 上升趋势; 反之下降
    ema_12 = df['close'].ewm(span=12, adjust=False).mean()
    ema_26 = df['close'].ewm(span=26, adjust=False).mean()
    df['trend_12_26'] = (ema_12 - ema_26) / (df['close'] + 1e-8)

    # ========== 8. MACD柱 ==========
    # MACD柱 > 0 → 上升动能; < 0 → 下降动能
    macd_line = ema_12 - ema_26
    signal_line = macd_line.ewm(span=9, adjust=False).mean()
    df['macd_hist'] = macd_line - signal_line

    # ========== 9. ADX (趋势强度) ==========
    # ADX > 25 → 强趋势; < 20 → 震荡
    high = df['high']
    low = df['low']
    close = df['close']

    plus_dm = high.diff().clip(lower=0)
    minus_dm = (-low.diff()).clip(lower=0)
    tr = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low - close.shift(1)).abs()
    ], axis=1).max(axis=1)

    atr_14 = tr.rolling(14).mean()
    plus_di = 100 * plus_dm.rolling(14).mean() / (atr_14 + 1e-8)
    minus_di = 100 * minus_dm.rolling(14).mean() / (atr_14 + 1e-8)
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di + 1e-8)
    df['adx_14'] = dx.rolling(14).mean()
    df['atr_14'] = atr_14

    # ========== 10. 日内振幅与开收收益 ==========
    df['hl_range'] = (high - low) / (close + 1e-8)
    df['co_return'] = (close - df['open']) / (df['open'] + 1e-8)

    # ========== 11. 拐点特征（差分）==========
    # 一阶差分: 指标变化速度 → 捕捉拐点
    # 三阶差分: 指标变化加速度 → 提前预警
    for col in ['rsi_14', 'z_price_20', 'bb_pos', 'trend_12_26', 'vol_ratio', 'co_return']:
        if col in df.columns:
            df[f'd1_{col}'] = df[col].diff()
            df[f'd3_{col}'] = df[col].diff(3)

    # ========== 12. 均线位置 ==========
    # 价格相对于长期均线的位置 → 长期趋势状态
    sma_50 = df['close'].rolling(50).mean()
    sma_200 = df['close'].rolling(200).mean()
    df['price_vs_sma50'] = (df['close'] - sma_50) / (sma_50 + 1e-8)
    df['price_vs_sma200'] = (df['close'] - sma_200) / (sma_200 + 1e-8)
    df['sma50_vs_sma200'] = (sma_50 - sma_200) / (sma_200 + 1e-8)

    return df

print("✅ make_ml_features() 定义完成")
print(f"   共产出特征类别: 多尺度动量(8) + 波动状态(3) + 成交量(1) + 反转(4) + 趋势(3) + ADX/ATR(2)")
print(f"   + 日内(2) + 拐点差分(12) + 均线位置(3) = 38+ 维特征")

#### 🔬 特征工程演示：用模拟数据展示各类特征

由于这个notebook无法连接聚宽API，我们用**模拟数据**来演示特征计算的效果。

In [ ]:
# 生成模拟的沪深300日频数据（含牛熊转换、震荡、急跌场景）
np.random.seed(42)
n_days = 1200  # 约5年

dates = pd.bdate_range('2019-01-02', periods=n_days, freq='B')

# 模拟价格：几段不同走势拼接
segments = [
    (0, 200, 0.0008, 0.012),      # 慢牛
    (200, 350, -0.002, 0.025),     # 急跌（类似2020Q1疫情）
    (350, 600, 0.0012, 0.015),     # 反弹牛
    (600, 800, -0.0003, 0.018),    # 震荡
    (800, 950, 0.0015, 0.013),     # 牛市
    (950, 1100, -0.0015, 0.022),   # 下跌
    (1100, 1200, 0.0005, 0.010),   # 平稳
]

returns = np.zeros(n_days)
for start, end, mu, sigma in segments:
    returns[start:end] = np.random.normal(mu, sigma, end - start)

price = 3500 * np.exp(np.cumsum(returns))

# 构造OHLCV
sim_df = pd.DataFrame(index=dates[:n_days])
sim_df['close'] = price
sim_df['open'] = price * (1 + np.random.normal(0, 0.003, n_days))
sim_df['high'] = np.maximum(sim_df['open'], sim_df['close']) * (1 + np.abs(np.random.normal(0, 0.005, n_days)))
sim_df['low'] = np.minimum(sim_df['open'], sim_df['close']) * (1 - np.abs(np.random.normal(0, 0.005, n_days)))
sim_df['volume'] = np.random.lognormal(20, 0.5, n_days)

# 应用特征工程
df_feat = make_ml_features(sim_df)

print(f"原始数据: {sim_df.shape[0]}行 × {sim_df.shape[1]}列")
print(f"特征工程后: {df_feat.shape[0]}行 × {df_feat.shape[1]}列")
print(f"新增特征数: {df_feat.shape[1] - sim_df.shape[1]}")
print(f"\n所有特征列:")
feat_cols = [c for c in df_feat.columns if c not in ['open','high','low','close','volume']]
for i, col in enumerate(feat_cols, 1):
    print(f"  {i:2d}. {col}", end="\t\t" if i % 3 != 0 else "\n")
print()

In [ ]:
# 可视化：关键特征在不同市场状态下的表现
fig, axes = plt.subplots(4, 2, figsize=(18, 16), sharex=True)
fig.suptitle('ML特征在不同市场状态下的表现', fontsize=16, fontweight='bold', y=0.995)

# 1. 价格走势
ax = axes[0, 0]
ax.plot(df_feat.index, df_feat['close'], 'k-', linewidth=1.5)
sma50 = df_feat['close'].rolling(50).mean()
sma200 = df_feat['close'].rolling(200).mean()
ax.plot(df_feat.index, sma50, '--', color='#2E86AB', alpha=0.7, label='SMA50')
ax.plot(df_feat.index, sma200, '--', color='#A23B72', alpha=0.7, label='SMA200')
ax.set_title('价格 + 均线', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 2. 波动率比值（状态检测核心）
ax = axes[0, 1]
vol_ratio = df_feat['vol_ratio'].dropna()
ax.plot(vol_ratio.index, vol_ratio, color='#FF6B35', linewidth=1)
ax.axhline(y=1.3, color='red', linestyle='--', alpha=0.7, label=f'危机阈值={CRISIS_VOL_RATIO_THRESH}')
ax.axhline(y=1.0, color='gray', linestyle='-', alpha=0.3)
ax.fill_between(vol_ratio.index, 1.3, vol_ratio, where=(vol_ratio > 1.3), alpha=0.3, color='red')
ax.set_title('vol_ratio (短期/长期波动率)', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 3. Z-score均值偏离
ax = axes[1, 0]
z = df_feat['z_price_20'].dropna()
ax.plot(z.index, z, color='#2E86AB', linewidth=1)
ax.axhline(y=2, color='red', linestyle='--', alpha=0.5, label='超买 (+2σ)')
ax.axhline(y=-2, color='green', linestyle='--', alpha=0.5, label='超卖 (-2σ)')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.fill_between(z.index, 0, z, where=(z > 2), alpha=0.3, color='red')
ax.fill_between(z.index, 0, z, where=(z < -2), alpha=0.3, color='green')
ax.set_title('z_price_20 (均值偏离度)', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 4. RSI
ax = axes[1, 1]
rsi = df_feat['rsi_14'].dropna()
ax.plot(rsi.index, rsi, color='#8B5A8E', linewidth=1)
ax.axhline(y=70, color='red', linestyle='--', alpha=0.7, label='超买(70)')
ax.axhline(y=30, color='green', linestyle='--', alpha=0.7, label='超卖(30)')
ax.fill_between(rsi.index, 70, rsi, where=(rsi > 70), alpha=0.3, color='red')
ax.fill_between(rsi.index, 30, rsi, where=(rsi < 30), alpha=0.3, color='green')
ax.set_title('RSI_14', fontweight='bold')
ax.set_ylim(0, 100)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 5. 趋势指标
ax = axes[2, 0]
trend = df_feat['trend_12_26'].dropna()
ax.plot(trend.index, trend * 100, color='#28a745', linewidth=1)
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.5)
ax.fill_between(trend.index, 0, trend * 100, where=(trend > 0), alpha=0.3, color='green')
ax.fill_between(trend.index, 0, trend * 100, where=(trend < 0), alpha=0.3, color='red')
ax.set_title('trend_12_26 (EMA差/价格 %)', fontweight='bold')
ax.grid(True, alpha=0.3)

# 6. MACD柱
ax = axes[2, 1]
macd = df_feat['macd_hist'].dropna()
colors = ['green' if v > 0 else 'red' for v in macd.values]
ax.bar(macd.index, macd.values, color=colors, alpha=0.6, width=2)
ax.set_title('MACD柱 (动量加速度)', fontweight='bold')
ax.grid(True, alpha=0.3)

# 7. 布林带位置
ax = axes[3, 0]
bbpos = df_feat['bb_pos'].dropna()
ax.plot(bbpos.index, bbpos, color='#E63946', linewidth=1)
ax.axhline(y=1, color='red', linestyle='--', alpha=0.5, label='上轨')
ax.axhline(y=0, color='green', linestyle='--', alpha=0.5, label='下轨')
ax.axhline(y=0.5, color='gray', linestyle='-', alpha=0.3)
ax.set_title('bb_pos (布林带相对位置)', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 8. RSI拐点检测（d1_rsi_14）
ax = axes[3, 1]
d1_rsi = df_feat['d1_rsi_14'].dropna()
ax.plot(d1_rsi.index, d1_rsi, color='#4ECDC4', linewidth=0.8, alpha=0.7)
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.5)
ax.set_title('d1_rsi_14 (RSI一阶差分 → 拐点检测)', fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.2 目标变量：未来5日收益

预测目标是 **未来h=5日的累计收益率**（而非1日收益），原因：

- **降噪**：1日收益噪声极大，5日收益信噪比更高
- **匹配持仓周期**：调仓频率为5天，预测周期与之匹配
- **更稳定的信号**：减少模型被日频噪声误导

$$\text{target}_t = \frac{\text{close}_{t+5} - \text{close}_t}{\text{close}_t}$$

> ⚠️ **关键**：目标通过 `shift(-horizon)` 构造，训练时严格切分"过去→现在"，确保无前瞻偏差。

### 3.3 双模型集成

| 模型 | 类型 | 作用 | 权重 |
|:-----|:-----|:-----|:-----|
| **HuberRegressor** | 鲁棒线性回归 | 捕捉反转的线性规律 | 45% |
| **HistGradientBoostingRegressor** | 梯度提升树 | 捕捉非线性状态/阈值效应 | 55% |

为什么选这两个？
- **Huber**：对异常值鲁棒（ε=1.35），反转场景常有极端收益
- **HistGBR**：速度快（比传统GBR快5-10倍），擅长识别分段/阈值效应

$$\text{score} = 0.45 \times \hat{y}_{\text{Huber}} + 0.55 \times \hat{y}_{\text{HistGBR}}$$

### 3.4 Walk-Forward训练机制

```
时间 →→→→→→→→→→→→→→→→→→→→→→→→→→→→→→→
      [════ 训练窗口(3年) ════] [预测]     ← 第1次训练
           [════ 训练窗口(3年) ════] [预测] ← 第2次训练(10天后)
                ...滚动前进...
```

- 训练窗口固定3年（756天），每10天重新训练
- 严格OOS：训练数据 ∈ [t-756, t-5]，预测 t 时刻
- 模型缓存：未到重训日则复用上次模型

### 3.5 样本加权策略

In [ ]:
def _make_sample_weight(y_values, half_life=126, tail_weight=3.0, tail_q=0.85):
    """
    样本权重 = 时间衰减 × 极端收益加权
    
    1. 时间衰减：最近的样本权重更大
       w_time(age) = 0.5^(age / half_life)
       → 半年前的样本权重是现在的50%
    
    2. 极端收益加权：暴涨暴跌样本权重×3
       → 让模型更关注极端行情（这些才是风控关键）
    """
    n = len(y_values)
    
    # === 时间衰减 ===
    age = np.arange(n)[::-1]              # [n-1, n-2, ..., 1, 0] 最新=0
    w_time = 0.5 ** (age / half_life)     # 指数衰减
    w_time = w_time / w_time.mean()       # 归一化（均值=1）
    
    # === 极端收益加权 ===
    abs_y = np.abs(y_values)
    finite = np.isfinite(abs_y)
    if finite.any():
        thr = np.quantile(abs_y[finite], tail_q)  # 85%分位数
    else:
        thr = 0.0
    w_tail = np.where(abs_y >= thr, tail_weight, 1.0)  # 极端样本×3
    
    return w_time * w_tail

# ── 可视化样本权重 ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. 时间衰减权重
n = 756
age = np.arange(n)[::-1]
w_time = 0.5 ** (age / 126)
w_time = w_time / w_time.mean()

ax = axes[0]
ax.plot(range(n), w_time, color='#2E86AB', linewidth=2)
ax.axhline(y=1, color='gray', linestyle='--', alpha=0.5, label='均匀权重')
ax.axvline(x=n - 126, color='red', linestyle='--', alpha=0.5, label='半年前 (weight≈50%)')
ax.set_xlabel('样本顺序 (从远到近 →)')
ax.set_ylabel('权重')
ax.set_title('时间衰减权重 (half_life=126)', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 2. 极端收益加权
y_sim = np.random.normal(0, 0.03, n)
y_sim[100:110] = np.random.normal(-0.08, 0.02, 10)  # 模拟暴跌
y_sim[400:405] = np.random.normal(0.06, 0.01, 5)     # 模拟暴涨

abs_y = np.abs(y_sim)
thr = np.quantile(abs_y, 0.85)
w_tail = np.where(abs_y >= thr, 3.0, 1.0)

ax = axes[1]
ax.scatter(range(n), y_sim * 100, c=w_tail, cmap='RdYlGn_r', s=5, alpha=0.6)
ax.axhline(y=thr * 100, color='red', linestyle='--', alpha=0.7, label=f'85%分位 = {thr*100:.1f}%')
ax.axhline(y=-thr * 100, color='red', linestyle='--', alpha=0.7)
ax.set_xlabel('样本顺序')
ax.set_ylabel('5日收益率 (%)')
ax.set_title('极端收益加权 (红色=3× 绿色=1×)', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 3. 综合权重 = 时间 × 极端
w_combined = w_time * w_tail
ax = axes[2]
ax.plot(range(n), w_combined, color='#A23B72', linewidth=1, alpha=0.7)
ax.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('样本顺序 (从远到近 →)')
ax.set_ylabel('综合权重')
ax.set_title('综合权重 = 时间衰减 × 极端加权', fontweight='bold')
ax.grid(True, alpha=0.3)

plt.suptitle('样本加权策略可视化', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n关键数字:")
print(f"  半年前的样本, 时间权重 ≈ {0.5**(126/126):.1%} (衰减到50%)")
print(f"  极端样本(前15%) 权重 = {3.0}×")
print(f"  最近的极端样本: 综合权重 ≈ {w_time[-1] * 3.0:.1f}×")

### 3.6 ML训练与预测完整流程演示

下面用模拟数据完整走一遍ML的 **特征构造 → 训练 → 预测** 流程。

In [ ]:
# ═══════════════════════════════════════════════════════════
# ML完整Walk-Forward演示（简化版，用模拟数据）
# ═══════════════════════════════════════════════════════════

horizon = ML_HORIZON  # 5日

# 1) 构建目标：未来5日收益
df_feat['target'] = df_feat['close'].pct_change(horizon).shift(-horizon)

# 2) 选择特征列
drop_cols = {'open', 'high', 'low', 'close', 'volume', 'target'}
feature_cols = [c for c in df_feat.columns if c not in drop_cols
                and df_feat[c].dtype in ['float64', 'float32', 'int64', 'int32']]

df_ml = df_feat.dropna(subset=['target']).copy()
X = df_ml[feature_cols]
y = df_ml['target']

print(f"特征矩阵: {X.shape[0]}行 × {X.shape[1]}列")
print(f"目标变量: {y.shape[0]}个样本")
print(f"目标统计: mean={y.mean():.5f}, std={y.std():.4f}")

# 3) Walk-Forward预测
train_window = ML_TRAIN_WINDOW
retrain_every = ML_RETRAIN_EVERY

predictions = pd.Series(index=X.index, dtype=float)
n = len(X)
start = train_window

model_lin = None
model_tree = None

print(f"\n开始Walk-Forward (窗口={train_window}, 重训频率={retrain_every})...")
train_count = 0

for i in range(start, n, retrain_every):
    # 训练数据: [i-train_window, i)
    X_train = X.iloc[i - train_window:i]
    y_train = y.iloc[i - train_window:i]
    
    # 样本权重
    sw = _make_sample_weight(y_train.values, ML_HALF_LIFE, ML_TAIL_WEIGHT, ML_TAIL_QUANTILE)
    
    # 训练双模型
    model_lin = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc', StandardScaler()),
        ('mdl', HuberRegressor(epsilon=1.35, alpha=1e-4, max_iter=600))
    ])
    model_lin.fit(X_train, y_train, mdl__sample_weight=sw)
    
    model_tree = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('mdl', HistGradientBoostingRegressor(
            max_depth=3, learning_rate=0.05, max_iter=250,
            l2_regularization=1e-3, random_state=42
        ))
    ])
    model_tree.fit(X_train, y_train, mdl__sample_weight=sw)
    
    train_count += 1
    
    # 预测未来retrain_every天
    j = min(i + retrain_every, n)
    X_block = X.iloc[i:j]
    
    p_lin = model_lin.predict(X_block)
    p_tree = model_tree.predict(X_block)
    
    predictions.iloc[i:j] = ENSEMBLE_LIN_WEIGHT * p_lin + ENSEMBLE_TREE_WEIGHT * p_tree

print(f"✅ 完成! 共训练 {train_count} 次")
print(f"有效预测: {predictions.notna().sum()} / {len(predictions)} 个样本")

# 4) 评估: 预测 vs 实际的IC
valid = predictions.notna()
if valid.any():
    ic_spearman = predictions[valid].corr(y[valid], method='spearman')
    ic_pearson = predictions[valid].corr(y[valid], method='pearson')
    print(f"\nIC (Spearman): {ic_spearman:.4f}")
    print(f"IC (Pearson):  {ic_pearson:.4f}")
    print(f"命中率: {(np.sign(predictions[valid]) == np.sign(y[valid])).mean():.2%}")

In [ ]:
# 可视化ML预测结果
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

valid_mask = predictions.notna()
pred_valid = predictions[valid_mask]
y_valid = y[valid_mask]
close_valid = df_ml.loc[valid_mask, 'close']

# 1. 预测得分时间序列
ax = axes[0, 0]
ax.plot(pred_valid.index, pred_valid * 100, color='#2E86AB', linewidth=0.8, alpha=0.7, label='ML预测(5日收益%)')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.5)
ax.fill_between(pred_valid.index, 0, pred_valid * 100,
                where=(pred_valid > 0), alpha=0.2, color='green')
ax.fill_between(pred_valid.index, 0, pred_valid * 100,
                where=(pred_valid < 0), alpha=0.2, color='red')
ax.set_title('ML预测得分 (market_score) 时间序列', fontweight='bold')
ax.set_ylabel('预测5日收益率 (%)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 2. 预测 vs 实际 散点图
ax = axes[0, 1]
ax.scatter(pred_valid * 100, y_valid * 100, alpha=0.3, s=8, color='steelblue')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
# 拟合线
z = np.polyfit(pred_valid, y_valid, 1)
p = np.poly1d(z)
x_line = np.linspace(pred_valid.min(), pred_valid.max(), 100)
ax.plot(x_line * 100, p(x_line) * 100, 'r--', linewidth=2, alpha=0.7)
ax.set_xlabel('预测收益率 (%)')
ax.set_ylabel('实际收益率 (%)')
ax.set_title(f'预测 vs 实际 (IC={ic_spearman:.4f})', fontweight='bold')
ax.grid(True, alpha=0.3)

# 3. 滚动IC
ax = axes[1, 0]
rolling_ic = pred_valid.rolling(60).corr(y_valid)
ax.plot(rolling_ic.index, rolling_ic, color='#A23B72', linewidth=1)
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.5)
ax.axhline(y=ic_spearman, color='blue', linestyle='--', alpha=0.5, label=f'总IC={ic_spearman:.4f}')
ax.fill_between(rolling_ic.index, 0, rolling_ic,
                where=(rolling_ic > 0), alpha=0.2, color='green')
ax.fill_between(rolling_ic.index, 0, rolling_ic,
                where=(rolling_ic < 0), alpha=0.2, color='red')
ax.set_title('滚动IC (60日窗口)', fontweight='bold')
ax.set_ylabel('Pearson IC')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 4. 预测得分分布
ax = axes[1, 1]
ax.hist(pred_valid * 100, bins=50, density=True, alpha=0.7, color='#2E86AB', edgecolor='black', linewidth=0.5)
ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax.set_xlabel('预测5日收益率 (%)')
ax.set_ylabel('密度')
ax.set_title(f'预测得分分布 (mean={pred_valid.mean()*100:.3f}%, std={pred_valid.std()*100:.3f}%)', fontweight='bold')
ax.grid(True, alpha=0.3)

plt.suptitle('ML市场状态预测 — Walk-Forward评估', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. 模块二：连续仓位映射

V1的仓位管理是**三档离散**的（100%/50%/0%），V2升级为**连续映射**，核心是三步：

1. **Sigmoid映射**：将ML的原始得分映射到 [0, 1]
2. **危机衰减**：乘以危机因子，进一步降低仓位
3. **EMA平滑 + 限幅**：避免仓位剧烈跳变

### 4.1 Sigmoid映射函数

$$\text{mom\_pct} = \sigma(\text{score} \times \text{scale}) = \frac{1}{1 + e^{-\text{score} \times \text{scale}}}$$

- `score > 0` → 看多 → 仓位偏向100%
- `score < 0` → 看空 → 仓位偏向0%
- `scale` 控制陡峭程度：越大，越接近0/1二值化

In [ ]:
def sigmoid_map(score, scale=8.0, center=0.0):
    """
    将ML原始得分映射到 [0, 1]
    
    score > 0 → 偏向1（看多，满仓动量）
    score < 0 → 偏向0（看空，满仓防守）
    scale 越大 → 映射越陡峭（越像开关）
    """
    x = (score - center) * scale
    x = np.clip(x, -20, 20)  # 防溢出
    return 1.0 / (1.0 + np.exp(-x))


def smooth_position(target_pct, current_pct, alpha=0.3, max_change=0.25):
    """
    EMA平滑 + 最大变化限幅
    
    alpha: 平滑系数（0.3=适中, 越小越平滑）
    max_change: 单次调仓最大变化（0.25=最多变25%）
    
    目的: 避免模型一个噪声预测导致仓位剧烈跳变
    """
    # EMA平滑: new = alpha * target + (1-alpha) * current
    smoothed = alpha * target_pct + (1 - alpha) * current_pct
    
    # 限幅: 单次变化不超过 max_change
    change = smoothed - current_pct
    if abs(change) > max_change:
        smoothed = current_pct + np.sign(change) * max_change
    
    return np.clip(smoothed, 0.0, 1.0)


# ═══════════════════════════════════════════════════════════
# 可视化Sigmoid映射
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. 不同scale的Sigmoid曲线
ax = axes[0]
scores = np.linspace(-0.1, 0.1, 200)
for scale, color, ls in [(5, '#2E86AB', '-'), (8, '#A23B72', '-'), (12, '#28a745', '-')]:
    pcts = [sigmoid_map(s, scale=scale) for s in scores]
    ax.plot(scores * 100, pcts, color=color, linewidth=2.5, linestyle=ls, label=f'scale={scale}')

# V1三档对比
ax.axhline(y=1.0, color='gray', linestyle=':', alpha=0.3)
ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.3)
ax.axhline(y=0.0, color='gray', linestyle=':', alpha=0.3)
ax.text(8, 1.02, 'V1: risk-on (100%)', fontsize=8, color='gray')
ax.text(8, 0.52, 'V1: neutral (50%)', fontsize=8, color='gray')
ax.text(8, 0.02, 'V1: risk-off (0%)', fontsize=8, color='gray')

ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.3)
ax.axvline(x=0, color='red', linestyle='--', alpha=0.3)
ax.set_xlabel('ML预测得分 (%)')
ax.set_ylabel('动量仓位比例')
ax.set_title('Sigmoid映射: 得分→仓位', fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.1)

# 2. 连续仓位 vs V1三档
ax = axes[1]
# 模拟一段market_score序列
np.random.seed(123)
n_sim = 100
scores_sim = np.cumsum(np.random.normal(0, 0.005, n_sim))

# V2连续仓位
v2_pct = [sigmoid_map(s, scale=8.0) for s in scores_sim]

# V1三档（简化模拟）
v1_pct = []
for s in scores_sim:
    if s > 0.02:
        v1_pct.append(1.0)
    elif s > -0.02:
        v1_pct.append(0.5)
    else:
        v1_pct.append(0.0)

ax.step(range(n_sim), v1_pct, color='#A23B72', linewidth=2, where='post', label='V1: 三档离散', alpha=0.7)
ax.plot(range(n_sim), v2_pct, color='#2E86AB', linewidth=2, label='V2: 连续映射')
ax.set_xlabel('调仓日')
ax.set_ylabel('动量仓位比例')
ax.set_title('V1三档 vs V2连续仓位', fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.1)

# 3. EMA平滑效果
ax = axes[2]
raw = [sigmoid_map(s, scale=8.0) for s in scores_sim]

# 模拟平滑过程
smoothed = [0.5]
for r in raw[1:]:
    s = smooth_position(r, smoothed[-1], alpha=0.3, max_change=0.25)
    smoothed.append(s)

ax.plot(range(n_sim), raw, color='#FF6B35', linewidth=1, alpha=0.6, label='原始(未平滑)')
ax.plot(range(n_sim), smoothed, color='#2E86AB', linewidth=2.5, label=f'平滑后 (α={POS_SMOOTH_ALPHA}, 限幅={POS_MAX_CHANGE})')
ax.set_xlabel('调仓日')
ax.set_ylabel('动量仓位比例')
ax.set_title('EMA平滑 + 变化限幅效果', fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.1)

plt.suptitle('连续仓位映射模块详解', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 关键区别:")
print("  V1: 三档离散切换，每次变化50%或100%，需要冷却期防抖")
print("  V2: 连续映射 + EMA平滑，仓位变化平滑，自然防抖")

## 5. 模块三：多因子复合动量选股

V1只用单一的 `MOM_20`（20日收益率）做选股排名。V2升级为**6因子复合**，截面Z-score标准化后加权合成。

### 5.1 六因子详解

| 因子 | 公式 | 捕捉什么 | 权重 |
|:-----|:-----|:---------|:-----|
| **mom_20** | $\frac{close_t}{close_{t-20}} - 1$ | 中短期价格趋势 | 25% |
| **mom_60** | $\frac{close_t}{close_{t-60}} - 1$ | 中期趋势确认 | 20% |
| **roc_10** | $\frac{close_t - close_{t-10}}{close_{t-10}}$ | 短期爆发力 | 15% |
| **rsi_14** | $100 - \frac{100}{1+RS}$ | 相对强弱（Z-score后） | 15% |
| **macd_h** | $MACD线 - Signal线$ | 趋势加速度 | 15% |
| **trend** | $\frac{EMA_{12} - EMA_{26}}{price}$ | 趋势方向 | 10% |

### 5.2 截面Z-score标准化

$$z_{i,f} = \frac{x_{i,f} - \bar{x}_f}{\sigma_f}$$

在每个调仓日，对20只ETF的每个因子做截面标准化（均值0、标准差1），然后加权合成：

$$\text{composite}_i = \sum_{f} w_f \cdot z_{i,f}$$

> **为什么要标准化？** 各因子量纲不同（RSI∈[0,100], MOM∈[-0.3,0.3], MACD∈[-50,50]），直接加权会被大数值因子主导。

In [ ]:
def _calc_ema(arr, span):
    """手动计算EMA (不依赖pandas, 聚宽兼容)"""
    alpha = 2.0 / (span + 1)
    result = np.zeros(len(arr))
    result[0] = arr[0]
    for i in range(1, len(arr)):
        result[i] = alpha * arr[i] + (1 - alpha) * result[i - 1]
    return result


def calc_composite_momentum_demo(etf_prices_dict, factor_weights):
    """
    多因子复合动量（演示版）
    
    输入: etf_prices_dict = {'ETF_A': np.array([...]), 'ETF_B': ...}
    输出: pd.Series — 每只ETF的复合动量得分
    
    步骤:
    1. 对每只ETF计算6个因子
    2. 截面Z-score标准化
    3. 加权合成
    """
    factor_data = {}
    
    for code, closes in etf_prices_dict.items():
        c = closes[:-1]  # 避免前瞻偏差: 用t-1收盘
        if len(c) < 62:
            continue
        
        factors = {}
        
        # ① mom_20: 20日收益率
        if len(c) >= 21:
            factors['mom_20'] = c[-1] / c[-21] - 1.0
        
        # ② mom_60: 60日收益率
        if len(c) >= 61:
            factors['mom_60'] = c[-1] / c[-61] - 1.0
        
        # ③ roc_10: 10日变动率
        if len(c) >= 11:
            factors['roc_10'] = (c[-1] - c[-11]) / (c[-11] + 1e-8)
        
        # ④ rsi_14: 相对强弱指标
        if len(c) >= 15:
            deltas = np.diff(c[-15:])
            gain = np.mean(deltas[deltas > 0]) if np.any(deltas > 0) else 0
            loss = -np.mean(deltas[deltas < 0]) if np.any(deltas < 0) else 0
            rs = gain / (loss + 1e-8)
            factors['rsi_14'] = 100 - 100 / (1 + rs)
        
        # ⑤ macd_h: MACD柱状图
        if len(c) >= 35:
            ema12 = _calc_ema(c, 12)
            ema26 = _calc_ema(c, 26)
            macd_line = ema12 - ema26
            signal = _calc_ema(macd_line[-9:], 9) if len(macd_line) >= 9 else macd_line[-1:]
            factors['macd_h'] = macd_line[-1] - signal[-1]
        
        # ⑥ trend: EMA趋势差
        if len(c) >= 27:
            ema12 = _calc_ema(c, 12)
            ema26 = _calc_ema(c, 26)
            factors['trend'] = (ema12[-1] - ema26[-1]) / (c[-1] + 1e-8)
        
        factor_data[code] = factors
    
    if not factor_data:
        return pd.Series(dtype=float)
    
    # 构建因子矩阵
    factor_df = pd.DataFrame(factor_data).T
    
    # 截面Z-score标准化
    factor_z = (factor_df - factor_df.mean()) / (factor_df.std() + 1e-8)
    
    # 加权合成
    composite = pd.Series(0.0, index=factor_z.index)
    for factor_name, weight in factor_weights.items():
        if factor_name in factor_z.columns:
            composite += weight * factor_z[factor_name].fillna(0)
    
    return composite, factor_df, factor_z


# ═══════════════════════════════════════════════════════════
# 演示：模拟20只ETF的多因子选股
# ═══════════════════════════════════════════════════════════
np.random.seed(42)
etf_names = [f'ETF_{i:02d}' for i in range(20)]

# 模拟不同走势的ETF
etf_prices = {}
for i, name in enumerate(etf_names):
    base = 1.0 + 0.1 * np.random.randn()
    drift = np.random.normal(0.0005 * (i - 10), 0.001)  # 有些涨有些跌
    vol = 0.01 + 0.005 * np.random.rand()
    returns = np.random.normal(drift, vol, 70)
    etf_prices[name] = base * np.exp(np.cumsum(returns))

# 计算复合动量
composite, factor_raw, factor_z = calc_composite_momentum_demo(etf_prices, FACTOR_WEIGHTS)

# 排名选Top10
ranking = composite.sort_values(ascending=False)
top10 = ranking.head(TOPK)

print("═" * 70)
print(f"{'ETF':<10} {'复合动量':>10} {'排名':>6} {'入选':>6}")
print("─" * 70)
for i, (etf, score) in enumerate(ranking.items(), 1):
    selected = '✅' if etf in top10.index else ''
    print(f"{etf:<10} {score:>10.4f} {i:>6} {selected:>6}")
print("═" * 70)

In [ ]:
# ═══════════════════════════════════════════════════════════
# 可视化：因子热力图 + 排名柱状图
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

# 图1：原始因子值热力图
ax = axes[0]
sns.heatmap(factor_raw.loc[ranking.index], annot=True, fmt='.3f', 
            cmap='RdYlGn', center=0, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('原始因子值', fontsize=14, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('ETF（按复合动量排序）')

# 图2：Z-score标准化后热力图
ax = axes[1]
sns.heatmap(factor_z.loc[ranking.index], annot=True, fmt='.2f',
            cmap='RdYlBu_r', center=0, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('截面Z-Score标准化', fontsize=14, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('')

# 图3：复合动量得分柱状图
ax = axes[2]
colors = ['#2ecc71' if etf in top10.index else '#95a5a6' for etf in ranking.index]
bars = ax.barh(range(len(ranking)), ranking.values, color=colors)
ax.set_yticks(range(len(ranking)))
ax.set_yticklabels(ranking.index)
ax.invert_yaxis()
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set_title(f'复合动量得分 (Top{TOPK}=绿色)', fontsize=14, fontweight='bold')
ax.set_xlabel('复合动量得分')

# 标注Top10分界线
if len(ranking) > TOPK:
    ax.axhline(y=TOPK - 0.5, color='red', linestyle='--', alpha=0.7, linewidth=2)
    ax.text(ax.get_xlim()[1] * 0.7, TOPK - 0.3, f'Top{TOPK}线', 
            color='red', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.suptitle('多因子复合动量选股过程', fontsize=16, fontweight='bold', y=1.02)
plt.show()

# 打印因子权重贡献
print("\n📊 因子权重配置:")
for f, w in FACTOR_WEIGHTS.items():
    print(f"  {f:<10} → {w:.0%}")
print(f"\n🏆 入选ETF: {', '.join(top10.index.tolist())}")

## 6. 模块四：风险控制体系 — 危机过滤器 + ATR止损

V2 的风控体系分为两个层次：

### 6.1 危机过滤器 (`calc_crisis_factor`)

**核心思想**：当市场波动率飙升且趋势向下时，自动压缩风险资产仓位。

| 触发条件 | 阈值 | 含义 |
|----------|------|------|
| `vol_ratio > VOL_SPIKE_THRESHOLD (1.3)` | 短期波动率/长期波动率 > 1.3 | 波动异常放大 |
| `trend < TREND_NEG_THRESHOLD (-0.01)` | EMA12 < EMA26 且差距 > 1% | 趋势确认下行 |

**两条件同时满足** → 计算 `severity`（严重程度），映射为阻尼因子 $d \in [0.3, 1.0]$：

$$d = \max(0.3, \; 1.0 - \text{severity})$$

$$\text{severity} = \frac{\text{vol\_ratio} - 1.3}{1.3} + |\text{trend}| \times 10$$

### 6.2 ATR 止损 (`check_atr_stop`)

**原理**：基于资产自身的波动幅度（ATR）动态设定止损线。

$$\text{stop\_price} = \text{highest\_since\_buy} \times (1 - \text{ATR\_MULTI} \times \text{ATR})$$

- `ATR_MULTI = 3.0`：允许3倍ATR的回撤空间
- ATR周期 = 14天

**相比V1的改进**：
- V1：固定阈值二元切换 → 要么全仓要么不持仓
- V2：连续调节 + 个股止损 → 更精细的风险控制

In [ ]:
# ═══════════════════════════════════════════════════════════
# 6.1 危机过滤器 — calc_crisis_factor
# ═══════════════════════════════════════════════════════════

VOL_SPIKE_THRESHOLD = 1.3
TREND_NEG_THRESHOLD = -0.01

def calc_crisis_factor(closes_benchmark, vol_spike_thr=VOL_SPIKE_THRESHOLD,
                       trend_neg_thr=TREND_NEG_THRESHOLD):
    """
    危机过滤器：返回 damping factor ∈ [0.3, 1.0]
    
    1.0 = 市场正常，不干预
    <1.0 = 市场危机，压缩仓位
    0.3 = 极端危机，最低保留30%仓位
    """
    c = np.array(closes_benchmark, dtype=float)
    
    # 波动率比: 短期(10日) / 长期(60日)
    if len(c) < 61:
        return 1.0, {}
    
    rets = np.diff(np.log(c))
    vol_short = np.std(rets[-10:]) * np.sqrt(252)
    vol_long  = np.std(rets[-60:]) * np.sqrt(252)
    vol_ratio = vol_short / (vol_long + 1e-8)
    
    # 趋势: EMA12 vs EMA26
    ema12 = _calc_ema(c, 12)
    ema26 = _calc_ema(c, 26)
    trend = (ema12[-1] - ema26[-1]) / (c[-1] + 1e-8)
    
    # 判断危机
    if vol_ratio > vol_spike_thr and trend < trend_neg_thr:
        severity = (vol_ratio - vol_spike_thr) / vol_spike_thr + abs(trend) * 10
        damping = max(0.3, 1.0 - severity)
    else:
        damping = 1.0
        severity = 0.0
    
    info = {
        'vol_ratio': vol_ratio,
        'trend': trend,
        'severity': severity,
        'damping': damping,
        'crisis': damping < 1.0
    }
    return damping, info


# ═══════════════════════════════════════════════════════════
# 6.2 ATR止损 — check_atr_stop
# ═══════════════════════════════════════════════════════════

ATR_MULTI = 3.0
ATR_PERIOD = 14

def calc_atr(highs, lows, closes, period=ATR_PERIOD):
    """计算Average True Range"""
    n = len(closes)
    if n < period + 1:
        return np.nan
    
    tr_list = []
    for i in range(1, n):
        tr = max(highs[i] - lows[i],
                 abs(highs[i] - closes[i-1]),
                 abs(lows[i] - closes[i-1]))
        tr_list.append(tr)
    
    # 简单移动平均ATR
    return np.mean(tr_list[-period:])


def check_atr_stop(highest_since_buy, current_price, atr_value, 
                   atr_multi=ATR_MULTI):
    """
    ATR止损检查
    
    返回: (should_stop, stop_price, distance_pct)
    """
    if np.isnan(atr_value) or atr_value <= 0:
        return False, np.nan, np.nan
    
    stop_price = highest_since_buy * (1 - atr_multi * atr_value / highest_since_buy)
    should_stop = current_price < stop_price
    distance_pct = (current_price - stop_price) / highest_since_buy
    
    return should_stop, stop_price, distance_pct


# ═══════════════════════════════════════════════════════════
# 演示：在模拟数据上展示危机检测
# ═══════════════════════════════════════════════════════════

# 使用之前生成的模拟基准数据（含牛市/熊市/暴跌/盘整）
sim_damping = []
sim_vol_ratios = []
sim_trends = []
sim_crisis = []

for t in range(61, len(sim_close)):
    d, info = calc_crisis_factor(sim_close[:t+1])
    sim_damping.append(d)
    sim_vol_ratios.append(info['vol_ratio'])
    sim_trends.append(info['trend'])
    sim_crisis.append(info['crisis'])

idx = np.arange(61, len(sim_close))

print(f"总交易日: {len(sim_damping)}")
print(f"危机天数: {sum(sim_crisis)} ({sum(sim_crisis)/len(sim_crisis)*100:.1f}%)")
print(f"阻尼因子范围: [{min(sim_damping):.3f}, {max(sim_damping):.3f}]")

In [ ]:
# ═══════════════════════════════════════════════════════════
# 可视化：危机过滤器 + ATR止损
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)

# 图1: 基准价格 + 危机区间标注
ax = axes[0]
ax.plot(sim_close, color='steelblue', linewidth=1)
crisis_mask = np.array(sim_crisis)
crisis_starts = []
crisis_ends = []
in_crisis = False
for i, c in enumerate(crisis_mask):
    if c and not in_crisis:
        crisis_starts.append(i + 61)
        in_crisis = True
    elif not c and in_crisis:
        crisis_ends.append(i + 61)
        in_crisis = False
if in_crisis:
    crisis_ends.append(len(sim_close) - 1)
for s, e in zip(crisis_starts, crisis_ends):
    ax.axvspan(s, e, alpha=0.3, color='red')
ax.set_ylabel('基准价格')
ax.set_title('基准走势 + 危机区间（红色阴影）', fontsize=13, fontweight='bold')
ax.legend(['基准价格', '危机区间'], loc='upper left')

# 图2: 波动率比
ax = axes[1]
ax.plot(idx, sim_vol_ratios, color='darkorange', linewidth=1)
ax.axhline(y=VOL_SPIKE_THRESHOLD, color='red', linestyle='--', 
           label=f'阈值 = {VOL_SPIKE_THRESHOLD}')
ax.fill_between(idx, VOL_SPIKE_THRESHOLD, sim_vol_ratios, 
                where=[v > VOL_SPIKE_THRESHOLD for v in sim_vol_ratios],
                alpha=0.3, color='red')
ax.set_ylabel('Vol Ratio')
ax.set_title('波动率比 (短期/长期)', fontsize=13, fontweight='bold')
ax.legend(loc='upper right')

# 图3: 趋势指标
ax = axes[2]
ax.plot(idx, sim_trends, color='purple', linewidth=1)
ax.axhline(y=TREND_NEG_THRESHOLD, color='red', linestyle='--',
           label=f'阈值 = {TREND_NEG_THRESHOLD}')
ax.axhline(y=0, color='gray', linewidth=0.5)
ax.fill_between(idx, TREND_NEG_THRESHOLD, sim_trends,
                where=[t < TREND_NEG_THRESHOLD for t in sim_trends],
                alpha=0.3, color='red')
ax.set_ylabel('Trend')
ax.set_title('趋势指标 (EMA12 - EMA26) / Price', fontsize=13, fontweight='bold')
ax.legend(loc='upper right')

# 图4: 阻尼因子
ax = axes[3]
ax.fill_between(idx, 0, sim_damping, alpha=0.4, color='steelblue')
ax.plot(idx, sim_damping, color='steelblue', linewidth=1)
ax.axhline(y=1.0, color='green', linestyle='--', alpha=0.5, label='正常 (d=1.0)')
ax.axhline(y=0.3, color='red', linestyle='--', alpha=0.5, label='极端 (d=0.3)')
ax.set_ylabel('Damping Factor')
ax.set_xlabel('交易日')
ax.set_title('阻尼因子 d ∈ [0.3, 1.0]', fontsize=13, fontweight='bold')
ax.set_ylim(-0.05, 1.15)
ax.legend(loc='lower right')

plt.tight_layout()
plt.show()

# ATR止损示例
print("\n" + "═" * 60)
print("ATR止损示例:")
print("─" * 60)
# 模拟一只ETF的高开低收
np.random.seed(123)
fake_close = sim_close[-30:]
fake_high = fake_close * (1 + np.abs(np.random.normal(0, 0.005, len(fake_close))))
fake_low = fake_close * (1 - np.abs(np.random.normal(0, 0.005, len(fake_close))))
atr_val = calc_atr(fake_high, fake_low, fake_close, period=14)
highest = np.max(fake_close)
current = fake_close[-1]

stop, stop_px, dist = check_atr_stop(highest, current, atr_val)
print(f"  买入以来最高价: {highest:.4f}")
print(f"  当前价:         {current:.4f}")
print(f"  ATR(14):        {atr_val:.4f}")
print(f"  止损价:         {stop_px:.4f}  (最高价 × (1 - {ATR_MULTI}×ATR/最高价))")
print(f"  距离止损:       {dist*100:.2f}%")
print(f"  是否触发止损:   {'⚠️ 是' if stop else '✅ 否'}")

## 7. 主调仓流程 — `rebalance()` 五步走

V2 策略的核心逻辑在 `rebalance()` 函数中，每个调仓日执行以下 **5 个步骤**：

```
┌─────────────────────────────────────────────────────────────┐
│                    rebalance() 五步流程                       │
│                                                             │
│  Step 1 ─── ML预测 ──→ ml_score ∈ [0, 1]                   │
│       │         └─ make_ml_features → dual model → sigmoid  │
│       ▼                                                     │
│  Step 2 ─── 平滑仓位 ──→ ml_pos ∈ [0, 1]                   │
│       │         └─ EMA平滑 + 最大变化限制(25%)               │
│       ▼                                                     │
│  Step 3 ─── 危机过滤 ──→ crisis_damping ∈ [0.3, 1.0]       │
│       │         └─ vol_ratio + trend → damping              │
│       ▼                                                     │
│  Step 4 ─── 选股 + 权重                                     │
│       │         └─ 复合动量 → TopK → 等权                    │
│       │         └─ risky = ml_pos × damping                 │
│       │         └─ 避险 = 1 - risky                          │
│       ▼                                                     │
│  Step 5 ─── 执行 + 止损                                     │
│             └─ ATR止损检查 → apply_target_weights            │
└─────────────────────────────────────────────────────────────┘
```

### 关键公式

**风险资产总仓位**：
$$w_{\text{risky}} = \text{ml\_pos} \times \text{crisis\_damping}$$

**单只ETF权重**（等权分配）：
$$w_i = \frac{w_{\text{risky}}}{\text{TopK}}$$

**避险资产仓位**：
$$w_{\text{defensive}} = 1 - w_{\text{risky}}$$

### 仓位组合示例

| 场景 | ML得分 | ml_pos | damping | risky仓位 | 避险仓位 | 单只ETF |
|------|--------|--------|---------|-----------|----------|---------|
| 牛市确认 | 0.8 | 0.88 | 1.0 | 88% | 12% | 8.8% |
| 震荡市 | 0.5 | 0.50 | 1.0 | 50% | 50% | 5.0% |
| 温和下跌 | 0.3 | 0.27 | 0.85 | 23% | 77% | 2.3% |
| 暴跌危机 | 0.2 | 0.12 | 0.40 | 4.8% | 95.2% | 0.5% |

In [ ]:
# ═══════════════════════════════════════════════════════════
# 端到端模拟：完整的 rebalance 流程
# ═══════════════════════════════════════════════════════════

def simulate_full_rebalance(sim_close, etf_prices, day_idx, 
                            prev_ml_pos, g_state):
    """
    模拟一次完整的rebalance调用
    
    参数:
        sim_close: 基准收盘价序列
        etf_prices: dict of ETF prices
        day_idx: 当前日索引
        prev_ml_pos: 上一期ML仓位
        g_state: 全局状态字典
    """
    result = {}
    
    # ──── Step 1: ML预测 ────
    # 在实际策略中这里会调用 ml_predict_market()
    # 这里用模拟分数代替
    np.random.seed(day_idx)
    trend_signal = (sim_close[day_idx] / sim_close[max(0, day_idx-60)] - 1)
    noise = np.random.normal(0, 0.1)
    raw_score = 0.5 + trend_signal * 5 + noise  # 模拟ML输出
    ml_score = sigmoid_map(raw_score, k=SIGMOID_K, x0=SIGMOID_X0)
    result['ml_score_raw'] = raw_score
    result['ml_score'] = ml_score
    
    # ──── Step 2: 平滑仓位 ────
    ml_pos = smooth_position(ml_score, prev_ml_pos,
                             ema_alpha=EMA_ALPHA, max_change=MAX_POS_CHANGE)
    result['ml_pos'] = ml_pos
    result['prev_ml_pos'] = prev_ml_pos
    
    # ──── Step 3: 危机过滤 ────
    damping, crisis_info = calc_crisis_factor(sim_close[:day_idx+1])
    result['damping'] = damping
    result['crisis'] = crisis_info.get('crisis', False)
    
    # ──── Step 4: 选股 + 权重计算 ────
    composite, _, _ = calc_composite_momentum_demo(
        {k: v[:day_idx+1] for k, v in etf_prices.items()},
        FACTOR_WEIGHTS
    )
    top_etfs = composite.sort_values(ascending=False).head(TOPK).index.tolist()
    
    risky_total = ml_pos * damping
    defensive_total = 1.0 - risky_total
    etf_weight = risky_total / TOPK if TOPK > 0 else 0
    
    result['risky_total'] = risky_total
    result['defensive_total'] = defensive_total
    result['etf_weight'] = etf_weight
    result['top_etfs'] = top_etfs
    
    # ──── Step 5: (ATR止损在实盘中执行) ────
    result['step5_note'] = 'ATR止损在持仓循环中检查'
    
    return result, ml_pos


# ═══════════════════════════════════════════════════════════
# 运行多期模拟
# ═══════════════════════════════════════════════════════════
rebalance_days = list(range(100, len(sim_close) - 10, REBALANCE_EVERY))
history = []
ml_pos_track = 0.5  # 初始仓位

for day in rebalance_days:
    res, ml_pos_track = simulate_full_rebalance(
        sim_close, etf_prices, min(day, 69),  # etf_prices只有70天
        ml_pos_track, {}
    )
    res['day'] = day
    history.append(res)

hist_df = pd.DataFrame(history)

print(f"模拟了 {len(history)} 个调仓日")
print(f"\n{'日期':>6} {'ML得分':>8} {'ML仓位':>8} {'阻尼':>6} {'风险仓位':>8} {'避险仓位':>8} {'危机':>4}")
print("─" * 60)
for _, row in hist_df.head(15).iterrows():
    print(f"{int(row['day']):>6} {row['ml_score']:>8.3f} {row['ml_pos']:>8.3f} "
          f"{row['damping']:>6.2f} {row['risky_total']:>8.1%} {row['defensive_total']:>8.1%} "
          f"{'⚠️' if row['crisis'] else '✅':>4}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# 可视化：调仓流程全景图
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(5, 1, figsize=(16, 16), sharex=True)

days = hist_df['day'].values

# 图1: 基准价格
ax = axes[0]
ax.plot(sim_close, color='steelblue', linewidth=1, alpha=0.7)
for d in days:
    ax.axvline(x=d, color='gray', alpha=0.1, linewidth=0.5)
ax.set_ylabel('基准价格')
ax.set_title('Step 0: 市场走势', fontsize=12, fontweight='bold')

# 图2: ML得分 → sigmoid映射
ax = axes[1]
ax.bar(days, hist_df['ml_score'].values, width=REBALANCE_EVERY*0.8, 
       color=['#2ecc71' if s > 0.5 else '#e74c3c' for s in hist_df['ml_score']],
       alpha=0.7)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_ylabel('ML Score')
ax.set_title('Step 1: ML预测 → Sigmoid映射', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1)

# 图3: 平滑后ML仓位
ax = axes[2]
ax.plot(days, hist_df['ml_pos'].values, 'o-', color='darkorange', 
        markersize=4, linewidth=1.5, label='平滑后仓位')
ax.plot(days, hist_df['ml_score'].values, 's--', color='gray', 
        markersize=3, linewidth=0.8, alpha=0.5, label='原始ML得分')
ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.3)
ax.set_ylabel('ML Position')
ax.set_title('Step 2: EMA平滑 + 变化限制', fontsize=12, fontweight='bold')
ax.legend(loc='upper right')
ax.set_ylim(0, 1)

# 图4: 阻尼因子
ax = axes[3]
colors_damp = ['#e74c3c' if d < 1 else '#2ecc71' for d in hist_df['damping']]
ax.bar(days, hist_df['damping'].values, width=REBALANCE_EVERY*0.8,
       color=colors_damp, alpha=0.7)
ax.axhline(y=1.0, color='green', linestyle='--', alpha=0.3)
ax.set_ylabel('Damping')
ax.set_title('Step 3: 危机阻尼因子', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.15)

# 图5: 最终仓位分配
ax = axes[4]
risky = hist_df['risky_total'].values
defensive = hist_df['defensive_total'].values
ax.fill_between(days, 0, risky, alpha=0.6, color='#3498db', label='风险资产', step='mid')
ax.fill_between(days, risky, risky + defensive, alpha=0.6, color='#f39c12', 
                label='避险资产', step='mid')
ax.set_ylabel('仓位比例')
ax.set_xlabel('交易日')
ax.set_title('Step 4-5: 最终仓位分配 (风险 + 避险 = 100%)', fontsize=12, fontweight='bold')
ax.legend(loc='upper right')
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.suptitle('V2策略 rebalance() 五步流程全景', fontsize=15, fontweight='bold', y=1.01)
plt.show()

## 8. V1 vs V2 全面对比

### 8.1 架构差异

| 维度 | V1 策略 | V2 策略 |
|------|---------|---------|
| **市场判断** | 规则型：MA + 波动率阈值 | ML型：HuberReg + HistGBR 双模型集成 |
| **信号生成** | 二元：risk-on / risk-off | 连续：sigmoid映射 ∈ [0, 1] |
| **仓位切换** | 离散跳变（0% ↔ 100%） | EMA平滑 + 最大变化25%限制 |
| **动量因子** | 单因子 MOM（N日收益率） | 6因子复合：MOM20/60 + ROC + RSI + MACD + Trend |
| **选股标准化** | 原始动量排序 | 截面Z-Score标准化后排序 |
| **风控机制** | 冷却期（固定N天不交易） | 危机过滤器（连续阻尼）+ ATR动态止损 |
| **避险仓位** | 0% 或 100%（全切） | 连续分配 ∈ [0%, 100%] |
| **训练方式** | 无 | Walk-Forward滚动训练，3年窗口 |
| **样本处理** | 无 | 时间衰减权重 + 尾部增强权重 |
| **再训练** | 无 | 每10个交易日自动再训练 |

### 8.2 优缺点分析

**V2 优势** ✅：
1. **避免大幅回撤时的猛烈仓位切换** — 平滑机制减少交易成本和滑点
2. **多因子选股更鲁棒** — 单一因子失效时其他因子可以补偿
3. **ML捕捉非线性特征** — 传统规则难以表达的复杂市场模式
4. **风控更精细** — 危机严重程度与仓位压缩成正比

**V2 劣势** ⚠️：
1. **模型复杂度高** — 过拟合风险增加
2. **计算量大** — Walk-Forward滚动训练消耗资源
3. **调参空间大** — 参数多（ML + 平滑 + 动量 + 风控）
4. **依赖历史模式** — ML模型在"前所未见"的市场中可能失效

### 8.3 建议使用场景

| 场景 | 推荐 |
|------|------|
| 快速验证策略想法 | V1 ✅ |
| 正式回测和实盘 | V2 ✅ |
| 计算资源有限 | V1 ✅ |
| 需要精细风控 | V2 ✅ |
| 学习理解策略逻辑 | V1 ✅ → V2 |

In [ ]:
# ═══════════════════════════════════════════════════════════
# V1 vs V2 仓位行为对比
# ═══════════════════════════════════════════════════════════

# 模拟V1的二元仓位决策
def v1_position(closes, ma_short=20, ma_long=60, vol_threshold=1.5):
    """V1简化版：MA交叉 + 波动率阈值 → 0/1"""
    if len(closes) < ma_long + 1:
        return 0.0
    ma_s = np.mean(closes[-ma_short:])
    ma_l = np.mean(closes[-ma_long:])
    rets = np.diff(np.log(closes[-61:]))
    vol_s = np.std(rets[-10:]) * np.sqrt(252)
    vol_l = np.std(rets[-60:]) * np.sqrt(252)
    vol_ratio = vol_s / (vol_l + 1e-8)
    
    if ma_s > ma_l and vol_ratio < vol_threshold:
        return 1.0  # Risk-on
    else:
        return 0.0  # Risk-off

# 计算V1和V2的仓位序列
v1_positions = []
v2_positions = []

for day in range(61, len(sim_close)):
    # V1
    v1_pos = v1_position(sim_close[:day+1])
    v1_positions.append(v1_pos)
    
    # V2 (已在history中)
    pass

# V2从hist_df插值到每日
v2_daily = np.interp(
    np.arange(61, len(sim_close)),
    hist_df['day'].values,
    hist_df['risky_total'].values
)

# 可视化对比
fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

idx_daily = np.arange(61, len(sim_close))

# 图1: 市场走势
ax = axes[0]
ax.plot(sim_close, color='steelblue', linewidth=1)
ax.set_ylabel('基准价格')
ax.set_title('市场走势', fontsize=13, fontweight='bold')

# 图2: V1仓位 (离散)
ax = axes[1]
ax.fill_between(idx_daily, 0, v1_positions, alpha=0.5, color='#e74c3c', step='post')
ax.set_ylabel('风险资产仓位')
ax.set_title('V1: 二元切换 (0% 或 100%)', fontsize=13, fontweight='bold')
ax.set_ylim(-0.05, 1.1)
# 统计切换次数
switches_v1 = sum(1 for i in range(1, len(v1_positions)) 
                  if v1_positions[i] != v1_positions[i-1])
ax.text(0.98, 0.85, f'切换次数: {switches_v1}', transform=ax.transAxes,
        ha='right', fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# 图3: V2仓位 (连续)
ax = axes[2]
ax.fill_between(idx_daily, 0, v2_daily, alpha=0.5, color='#3498db')
ax.set_ylabel('风险资产仓位')
ax.set_xlabel('交易日')
ax.set_title('V2: 连续调节 (0% ~ 100%)', fontsize=13, fontweight='bold')
ax.set_ylim(-0.05, 1.1)
# 统计平均变化幅度
changes_v2 = np.abs(np.diff(v2_daily))
ax.text(0.98, 0.85, f'平均每日变化: {np.mean(changes_v2)*100:.2f}%', 
        transform=ax.transAxes, ha='right', fontsize=12,
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))

plt.tight_layout()
plt.suptitle('V1 vs V2 仓位行为对比', fontsize=15, fontweight='bold', y=1.01)
plt.show()

# 量化对比
print("\n📊 仓位行为量化对比:")
print(f"  V1 仓位切换次数: {switches_v1}")
print(f"  V1 平均仓位: {np.mean(v1_positions)*100:.1f}%")
print(f"  V2 平均仓位: {np.mean(v2_daily)*100:.1f}%")
print(f"  V2 仓位标准差: {np.std(v2_daily)*100:.1f}%")
print(f"  V2 最大单日变化: {np.max(changes_v2)*100:.2f}%")

## 9. 参数调优指南

### 9.1 参数分层

V2 策略的参数按重要性分为 **5 层**，建议从第一层开始调优：

| 层级 | 参数 | 默认值 | 调优范围 | 敏感度 |
|------|------|--------|----------|--------|
| **Tier 1: ML核心** | `RETRAIN_EVERY` | 10 | 5~20 | ⭐⭐⭐ |
| | `TRAIN_WINDOW` | 756 | 504~1008 | ⭐⭐⭐ |
| | `FWD_DAYS` | 5 | 3~10 | ⭐⭐⭐ |
| **Tier 2: 仓位映射** | `SIGMOID_K` | 3.0 | 2.0~5.0 | ⭐⭐ |
| | `EMA_ALPHA` | 0.3 | 0.1~0.5 | ⭐⭐ |
| | `MAX_POS_CHANGE` | 0.25 | 0.10~0.40 | ⭐⭐ |
| **Tier 3: 选股** | `TOPK` | 10 | 5~15 | ⭐⭐ |
| | `MOM_WIN` | 20 | 10~60 | ⭐ |
| **Tier 4: 风控** | `VOL_SPIKE_THRESHOLD` | 1.3 | 1.1~1.8 | ⭐⭐ |
| | `ATR_MULTI` | 3.0 | 2.0~4.0 | ⭐ |
| **Tier 5: 样本权重** | `HALF_LIFE` | 126 | 63~252 | ⭐ |
| | `TAIL_QUANTILE` | 0.85 | 0.80~0.95 | ⭐ |

### 9.2 推荐扫描组合

`V2_risk_scan.py` 提供了 **12组** 预设参数组合，覆盖以下策略风格：

| 组号 | 风格 | 关键参数差异 |
|------|------|-------------|
| 1-4 | 🏃 激进型 | Sigmoid K=2.5, TopK=12, VOL阈值=1.5 |
| 5-8 | ⚖️ 均衡型 | 默认参数，微调EMA和窗口 |
| 9-12 | 🛡️ 保守型 | Sigmoid K=4.0, TopK=8, VOL阈值=1.1 |

### 9.3 调参注意事项

⚠️ **避免过拟合**：
- Walk-Forward 训练窗口不宜太短（< 504天 ≈ 2年）
- 不要在同一数据集上反复调参
- 保留 out-of-sample 测试期

⚠️ **参数交互效应**：
- `SIGMOID_K` ↑ + `EMA_ALPHA` ↓ = 反应更快但更不稳定
- `VOL_SPIKE_THRESHOLD` ↓ + `ATR_MULTI` ↓ = 极度保守
- `TOPK` ↑ + `MOM_WIN` ↓ = 持仓分散但动量信号弱

In [ ]:
# ═══════════════════════════════════════════════════════════
# 参数敏感度分析：Sigmoid K 和 EMA_alpha 对仓位的影响
# ═══════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ── 图1: Sigmoid K 的影响 ──
ax = axes[0, 0]
x = np.linspace(-1, 2, 300)
for k_val in [1.5, 2.5, 3.0, 4.0, 5.0]:
    y = 1.0 / (1.0 + np.exp(-k_val * (x - 0.5)))
    ax.plot(x, y, label=f'K={k_val}', linewidth=2)
ax.axvline(x=0.5, color='gray', linestyle=':', alpha=0.3)
ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.3)
ax.set_xlabel('ML原始得分')
ax.set_ylabel('仓位')
ax.set_title('Sigmoid K 敏感度', fontsize=13, fontweight='bold')
ax.legend()
ax.set_xlim(-0.5, 1.5)

# ── 图2: EMA Alpha 平滑效果 ──
ax = axes[0, 1]
# 模拟一段波动的ML得分
np.random.seed(42)
raw_signal = 0.5 + 0.3 * np.sin(np.linspace(0, 4*np.pi, 100)) + np.random.normal(0, 0.1, 100)
raw_signal = np.clip(raw_signal, 0, 1)

ax.plot(raw_signal, color='gray', alpha=0.4, linewidth=1, label='原始信号')
for alpha_val in [0.1, 0.2, 0.3, 0.5]:
    smoothed = [raw_signal[0]]
    for i in range(1, len(raw_signal)):
        smoothed.append(alpha_val * raw_signal[i] + (1 - alpha_val) * smoothed[-1])
    ax.plot(smoothed, label=f'α={alpha_val}', linewidth=2)
ax.set_xlabel('时间步')
ax.set_ylabel('仓位')
ax.set_title('EMA Alpha 平滑效果', fontsize=13, fontweight='bold')
ax.legend()

# ── 图3: TopK 与集中度 ──
ax = axes[1, 0]
topk_range = range(3, 21)
single_weights = [1.0/k for k in topk_range]
hhi = [1.0/k for k in topk_range]  # 等权HHI = 1/K
ax.bar(list(topk_range), [w*100 for w in single_weights], color='steelblue', alpha=0.7)
ax.set_xlabel('TopK (持仓数量)')
ax.set_ylabel('单只ETF权重 (%)')
ax.set_title('TopK vs 单只ETF权重 (满仓时)', fontsize=13, fontweight='bold')
for k in [5, 10, 15]:
    ax.annotate(f'{100/k:.1f}%', xy=(k, 100/k), xytext=(k, 100/k+1.5),
                ha='center', fontweight='bold', color='red')

# ── 图4: VOL_SPIKE_THRESHOLD 的影响 ──
ax = axes[1, 1]
# 在模拟数据上测试不同阈值
thresholds = [1.1, 1.2, 1.3, 1.5, 1.8]
crisis_pcts = []
for thr in thresholds:
    crisis_count = 0
    for t in range(61, len(sim_close)):
        d, info = calc_crisis_factor(sim_close[:t+1], vol_spike_thr=thr)
        if d < 1.0:
            crisis_count += 1
    crisis_pcts.append(crisis_count / (len(sim_close) - 61) * 100)

colors_bar = ['#e74c3c' if p > 30 else '#f39c12' if p > 15 else '#2ecc71' 
              for p in crisis_pcts]
bars = ax.bar([str(t) for t in thresholds], crisis_pcts, color=colors_bar, alpha=0.7)
ax.set_xlabel('VOL_SPIKE_THRESHOLD')
ax.set_ylabel('危机触发天数占比 (%)')
ax.set_title('波动率阈值 vs 危机频率', fontsize=13, fontweight='bold')
for bar, pct in zip(bars, crisis_pcts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{pct:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.suptitle('参数敏感度分析', fontsize=15, fontweight='bold', y=1.01)
plt.show()

---

## 📝 总结

本 notebook 对 **V2_strategy.py** 进行了逐模块拆解：

| 模块 | 核心函数 | 作用 |
|------|---------|------|
| ML预测 | `ml_predict_market()` → `make_ml_features()` | 40+特征 → 双模型集成 → Walk-Forward |
| 仓位映射 | `sigmoid_map()` → `smooth_position()` | 连续 [0,1] 仓位 + EMA平滑 |
| 多因子选股 | `calc_composite_momentum()` | 6因子加权 + Z-score标准化 |
| 风险控制 | `calc_crisis_factor()` + `check_atr_stop()` | 危机阻尼 + ATR止损 |
| 主流程 | `rebalance()` | 五步走：ML→平滑→危机→选股→执行 |

### 后续方向
- 🔬 在聚宽回测平台运行 `V2_strategy.py` 验证实际表现
- 🔧 使用 `V2_risk_scan.py` 进行参数扫描
- 📊 对比 V1 和 V2 的回测结果（年化收益、最大回撤、夏普比率）
- 🧪 实验更多特征工程（如加入行业轮动、资金流向等因子）

---

*本notebook为 V2 策略代码详解文档，所有演示使用模拟数据，实际回测请在聚宽平台运行。*